In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/adnanik23/input-new')

In [ ]:
from gpt2_architecture import GPTModel

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
!pip3 install tiktoken

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import urllib.request
import ssl
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # Create an unverified SSL context
    ssl_context = ssl._create_unverified_context()

    # Downloading the file
    with urllib.request.urlopen(url, context=ssl_context) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # Unzipping the file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Add .tsv file extension
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


In [ ]:
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df.head()

In [ ]:
num_spam=df[df['Label']=='spam'].shape[0]
ham_subset=df[df['Label']=='ham'].sample(num_spam,random_state=123)
balanced_df=pd.concat([ham_subset,df[df['Label']=='spam']])
balanced_df["Label"]=balanced_df["Label"].map({'ham':0,'spam':1})

In [ ]:
def random_split(df,train_ratio,val_ratio):
  df=df.sample(frac=1,random_state=123).reset_index(drop=True)
  train_end=int(len(df)*train_ratio)
  val_end=train_end + int(len(df)*val_ratio)
  train_data=df[:train_end]
  val_data=df[train_end:val_end]
  test_data=df[val_end:]
  return train_data,val_data,test_data
train_df,val_df,test_df=random_split(balanced_df,0.7,0.1)

In [ ]:
train_df.to_csv("train.csv",index=None)
val_df.to_csv("val.csv",index=None)
test_df.to_csv("test.csv",index=None)

In [ ]:
from torch.utils.data import Dataset,DataLoader

In [ ]:
class CustomDataset(Dataset):
  def __init__(self,csv_path,tokenizer,max_length=None,eos_token_id=50256):
    self.data=pd.read_csv(csv_path)
    self.encoded_data=[tokenizer.encode(text) for text in self.data["Text"]]
    if max_length is None:
      self.max_length=self._longest_sequence_length()
    else:
      self.max_length=max_length
      self.encoded_data=[encoded_text[:self.max_length] for encoded_text in self.encoded_data]
    self.encoded_data=[encoded_text + [eos_token_id]*(self.max_length-len(encoded_text)) for encoded_text in self.encoded_data]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
     label=self.data.iloc[index]["Label"]
     input=self.encoded_data[index]
     label_tensor=torch.tensor(label,dtype=torch.long)
     input_tensor=torch.tensor(input,dtype=torch.long)
     return input_tensor,label_tensor

  def _longest_sequence_length(self):
    max_length=0
    for text in self.encoded_data:
      max_length=max(max_length,len(text))
    return max_length

In [ ]:
import tiktoken
tokenizer=tiktoken.get_encoding('gpt2')

In [ ]:
train_dataset=CustomDataset(csv_path='/kaggle/working/train.csv',tokenizer=tokenizer)
val_dataset=CustomDataset(csv_path='/kaggle/working/val.csv',tokenizer=tokenizer,max_length=train_dataset.max_length)
test_dataset=CustomDataset(csv_path='/kaggle/working/test.csv',tokenizer=tokenizer,max_length=train_dataset.max_length)

In [ ]:
train_loader=DataLoader(train_dataset,batch_size=8,shuffle=True,drop_last=True)
val_loader=DataLoader(val_dataset,batch_size=8,shuffle=True,drop_last=True)
test_loader=DataLoader(test_dataset,batch_size=8,shuffle=True,drop_last=True)

In [ ]:
from  gpt_download3 import download_and_load_gpt2

In [ ]:
settings,params=download_and_load_gpt2(model_size="124M",models_dir="gpt2")

In [ ]:
config={"vocab_size":50257,"context_length":1024,"embed_dim":768,"num_heads":12,"n_layers":12,"dropout":0.0,"qkv_bias":True}

In [ ]:
model=GPTModel(config)

In [ ]:
from gpt2_architecture import load_weights_into_model

In [ ]:
load_weights_into_model(model,params)

In [ ]:
model.eval()
for param in model.parameters():
  param.requires_grad=False
num_classes=2
model.final_nn=nn.Linear(config["embed_dim"],num_classes,bias=False)
for param in model.trf_blocks[-1].parameters():
  param.requires_grad=True
for param in model.final_layer_norm.parameters():
  param.requires_grad=True
model.to(device)

In [ ]:
def calc_accuracy_total(data_loader,model,device,num_batches=None):
  model.eval()
  correct_predictions,num_examples=0,0
  if num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      with torch.no_grad():
        output_batch=model(input_batch)[:,-1,:]
      predicted_labels=torch.argmax(output_batch,dim=-1)
      num_examples+=predicted_labels.shape[0]
      correct_predictions+=(predicted_labels==target_batch).sum().item()
    else:
      break
  return correct_predictions/num_examples

In [ ]:
def calc_loss_total(data_loader,model,device,num_batches=None):
  total_loss=0
  if len(data_loader)==0:
    return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      output_batch=model(input_batch)[:,-1,:]
      loss=torch.nn.functional.cross_entropy(output_batch,target_batch)
      total_loss+=loss.item()
    else:
      break
  return total_loss/num_batches

In [ ]:
def train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter):
  global_step=-1
  train_losses=[]
  val_losses=[]
  examples_seen=0
  for i in range(num_epochs):
    model.train()
    for i,(input_batch,target_batch) in enumerate(train_loader):
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      output_batch=model(input_batch)[:,-1,:]
      loss=torch.nn.functional.cross_entropy(output_batch,target_batch)
      loss.backward()
      optimizer.step()
      examples_seen += input_batch.shape[0]
      optimizer.zero_grad()
      global_step+=1
      if global_step%(eval_freq)==0:
        model.eval()
        with torch.no_grad():
            train_loss=calc_loss_total(train_loader,model=model,device=device,num_batches=eval_iter)
            val_loss=calc_loss_total(val_loader,model=model,device=device,num_batches=eval_iter)
            model.train()
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Ep {i+1} (step {global_step:06d}):" f" Train loss:{train_loss:.3f}, Val loss:{val_loss:.3f}")
  return train_losses,val_losses,examples_seen

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=5e-5,weight_decay=0.1)
num_epochs=5
train_losses,val_losses,examples_seen=train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq=50,eval_iter=5)

In [ ]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"Validation {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # Create a second x-axis for examples seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(examples_seen, train_values, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Examples seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))
plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

In [ ]:
print(calc_accuracy_total(train_loader,model,device))
print(calc_accuracy_total(val_loader,model,device))
print(calc_accuracy_total(test_loader,model,device))

In [ ]:
def inference(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # Prepare inputs to the model
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_embedding.weight.shape[0]
    # Note: In the book, this was originally written as pos_emb.weight.shape[1] by mistake
    # It didn't break the code but would have caused unnecessary truncation (to 768 instead of 1024)

    # Truncate sequences if they too long
    input_ids = input_ids[:min(max_length, supported_context_length)]

    # Pad sequences to the longest sequence
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # add batch dimension

    # Model inference
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # Logits of the last output token
    predicted_label = torch.argmax(logits, dim=-1).item()

    # Return the classified result
    return "spam" if predicted_label == 1 else "not spam"

In [ ]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(inference(
    text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

In [ ]:
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)

print(inference(
    text_2, model, tokenizer, device, max_length=train_dataset.max_length
))

In [ ]:
torch.save(model.state_dict(),"spam_classifier_weights.pth")